In [ ]:
from getpass import getpass

admin_rdm_url = 'https://admin.staging.rdm.example.com/'
cas_rdm_url = 'https://account.staging.rdm.example.com/'

idp_name_integrated_admin = None
idp_username_integrated_admin = None
idp_password_integrated_admin = None

user_guid = 'abcde'
user_name = 'test@example.com'

default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_name_integrated_admin is None:
    idp_name_integrated_admin = input(prompt=f'IdP Name for integrated admin: ')
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin}')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# 未ログインユーザが統合管理者向け、機関管理者向けのページにアクセスできないことを確認する

- サブシステム名: 管理者
- ページ/アドオン: 管理者
- 機能分類: 認証/認可
- シナリオ名: 未ログインユーザが統合管理者向け、機関管理者向けのページにアクセスできないことを確認する
- 用意するテストデータ:   
   ・GRDM 管理者のURL   
   ・CASのURL   
   ・アカウント(統合管理者）   
   ・未ログインユーザ    
   ・ユーザーのGUID（削除されていないユーザー）   
   ・ユーザー名（削除されていないユーザー）   
   ・登録プロバイダー(OSF Registries、少なくとも1件のRegistrationが紐付いている)   
   ・プレプリントプロバイダー(Open Science Framework、少なくとも1件のプレプリントが紐付いている)   
   ・コレクションプロバイダー(OSF Collection、少なくとも1件のプロジェクトが紐付いている)   
   ・USE_LOGIN_FORM が空（ENABLE_LOGIN_FORM が偽）の環境

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウで GRDM 管理者トップページを表示する

・`/account/login/` にリダイレクトされ、ログイン画面が表示されること   
・リダイレクト後のURLが「https://{admin_domain}/account/login/」であること

In [ ]:
import re
from urllib.parse import urlparse

async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

    login_url = page.url
    parsed_url = urlparse(login_url)
    assert re.fullmatch(
        r"/account/login/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {login_url}"

await run_pw(_step)

## アカウント(統合管理者)を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。   

※ ログインしてCSRFトークンを取得し、そのトークンを使用してテストケースNo.5、No.10、No.17、No.20、No.26を実施する


In [ ]:
csrf_token = None

async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin, idp_password_integrated_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

    cookies = await page.context.cookies()
    global csrf_token
    csrf_token = next((c['value'] for c in cookies if c['name'] == 'admin-csrf'), None)
    print(csrf_token)

await run_pw(_step)

## 管理者画面からログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    if await logout_btn.count() > 0:
        await logout_btn.click()
    else:
        await logout_link.click()

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ブラウザのアドレスバーに、「統合管理者」シートNo.5のリダイレクト後のURLを直接入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
import os

result_dir = default_result_path.split("notebooks/")[0] + "notebooks/"
template_result_dir = os.path.join(result_dir,'テスト手順-管理者機能-機関管理者の適切な権限-統合管理者')
url_list_file = os.path.join(template_result_dir, 'url_list.txt')

with open(url_list_file, 'r') as f:
    url_dict = dict(
        line.strip().split('=', 1)
        for line in f
        if '=' in line
    )

registration_provider_schemas_url = url_dict['registration_provider_schemas_url']
registration_provider_cannot_delete_url = url_dict['registration_provider_cannot_delete_url']
preprint_provider_cannot_delete_url = url_dict['preprint_provider_cannot_delete_url']
collection_provider_cannot_delete_url = url_dict['collection_provider_cannot_delete_url']

forbidden_permission_status = 403
forbidden_permission_message = "You do not have permission."

In [ ]:
async def _step(page):
    await page.goto(registration_provider_schemas_url)

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「統合管理者」シートNo.5のリダイレクト後のURLに対してPOSTメソッドを実行する。

HTTPステータスコードが 403 であること   
レスポンスボディに「You do not have permission.」というテキストが含まれていること

In [ ]:
async def _step(page):
    response = await page.request.post(
        registration_provider_schemas_url,
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer": registration_provider_schemas_url,
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
            "1": "on",
            "5": "on",
        },
        max_redirects=0,
    )

    assert response.status == forbidden_permission_status, (f"Expected HTTP status 403, but got {response.status}")

    response_body = await response.text()
    assert forbidden_permission_message in response_body, (
        f"Expected message '{forbidden_permission_message}' "
        f"not found in response body: {response_body}"
    )

await run_pw(_step)

## ブラウザのアドレスバーに、「統合管理者」シートNo.7のリダイレクト後のURLを直接入力する	

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(registration_provider_cannot_delete_url)

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「統合管理者」シートNo.10のリダイレクト後のURLを直接入力する	

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(preprint_provider_cannot_delete_url)

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「統合管理者」シートNo.13のリダイレクト後のURLを直接入力する	

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(collection_provider_cannot_delete_url)

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/management」を直接入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/management")

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「https://{admin_domain}/management/waffle_flag」に対してPOSTメソッドを実行する。

HTTPステータスコードが 403 であること   
レスポンスボディに「You do not have permission.」というテキストが含まれていること

In [ ]:
async def _step(page):
    response = await page.request.post(
        f"{admin_rdm_url.rstrip('/')}/management/waffle_flag",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "X-CSRFToken": csrf_token,
            "Referer": f"{admin_rdm_url.rstrip('/')}/management/waffle_flag",
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
        },
        max_redirects=0,
    )

    assert response.status == forbidden_permission_status, (
        f"Expected HTTP status 403, but got {response.status}"
    )

    response_body = await response.text()
    assert forbidden_permission_message in response_body, (
        f"Expected message '{forbidden_permission_message}' "
        f"not found in response body: {response_body}"
    )

await run_pw(_step)


##  ブラウザのアドレスバーに、「https://{admin_domain}/statistics/index」を直接入力する	

ページにアクセスできないこと   
「You do not have permission.」というテキストが表示されること

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/index")

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/quota_recalc」を直接入力する

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること 

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/quota_recalc")

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/quota_recalc/{ユーザーのGUID}」を直接入力する

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること 

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/quota_recalc/{user_guid}")

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics/test/mail」を直接入力する	

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること 

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/test/mail")

    await expect(page.get_by_role("heading", name=f"{forbidden_permission_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/Shibboleth.sso/Status」を直接入力する

ページにアクセスできないこと   
「You don't have permission to access this resource.」というテキストが表示されること

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/Shibboleth.sso/Status")

    await expect(page.get_by_role("heading", name="Forbidden")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("You don't have permission to access this resource.")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{cas_domain}/Shibboleth.sso/Status」を直接入力する

ページにアクセスできないこと   
「You don't have permission to access this resource.」というテキストが表示されること

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{cas_rdm_url.rstrip('/')}/Shibboleth.sso/Status")

    await expect(page.get_by_role("heading", name="Forbidden")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("You don't have permission to access this resource.")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「https://{admin_domain}/password/recover」に対してPOSTメソッドを実行する。

HTTPステータスコードが 404 であること   
レスポンスボディに「Resolver404」というテキストが含まれていること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される。

In [ ]:
not_found_status = 404
not_found_url_message = "Resolver404"

async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    response = await page.request.post(
        f"{admin_rdm_url.rstrip('/')}/password/recover",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer": f"{admin_rdm_url.rstrip('/')}/password/recover",
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
            "username": user_name,
        },
        max_redirects=0,
    )

    assert response.status == not_found_status, (
        f"Expected HTTP status 404, but got {response.status}"
    )

    response_body = await response.text()
    assert not_found_url_message in response_body, (
        f"Expected message '{not_found_url_message}' "
        f"not found in response body: {response_body}"
    )

await run_pw(_step)


## ブラウザのアドレスバーに、「https://{admin_domain}/password/recover/{signature}」を直接入力する

404ページが表示されること    
「Resolver404」というテキストが表示されること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される。

In [ ]:
signature = 'abcde'

async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/password/recover/{signature}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/password/reset/{token}/」を直接入力する

404ページが表示されること    
「Resolver404」というテキストが表示されること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される。

In [ ]:
token = 'dummytoken'

async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/password/reset/{token}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「https://{admin_domain}/password/reset/{token}」に対してPOSTメソッドを実行する。

HTTPステータスコードが 404 であること   
レスポンスボディに「Resolver404」というテキストが含まれていること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    fake_token = 'abc123-invalidtoken0000000'
    response = await page.request.post(
        f"{admin_rdm_url.rstrip('/')}/password/reset/{fake_token}/",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer": f"{admin_rdm_url.rstrip('/')}/password/reset/{fake_token}/",
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
            "password1": "TestPassword123!",
            "password2": "TestPassword123!",
        },
        max_redirects=0,
    )

    assert response.status == not_found_status, (
        f"Expected HTTP status 404, but got {response.status}"
    )

    response_body = await response.text()
    assert not_found_url_message in response_body, (
        f"Expected message '{not_found_url_message}' "
        f"not found in response body: {response_body}"
    )

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/password/reset/done」を直接入力する	

404ページが表示されること   
「Resolver404」というテキストが表示されること     
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/password/reset/done")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics/gather/{ts}/{signature}」を直接入力する

以下のJSONデータが表示されること   
{"state": "fail", "error": "access forbidden"}

In [ ]:
ts = 1

async def _step(page):
    print(f"{admin_rdm_url.rstrip('/')}/statistics/gather/{ts}/{signature}")
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/gather/{ts}/{signature}")

    response_text = await page.locator("body").inner_text()
    assert re.search(
        r'"state"\s*:\s*"fail"\s*,\s*"error"\s*:\s*"access forbidden"',
        response_text,
    ), f"期待するJSONデータが表示されていません。actual: {response_text}"

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics/gather/{access_token}」を直接入力する	

404ページが表示されること   
「Resolver404」というテキストが表示されること   

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/gather/{token}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/external_acc_update/{ts}/{signature}」を直接入力する

以下のJSONデータが表示されること   
{"state": "fail", "error": "access forbidden"}

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/external_acc_update/{ts}/{signature}")

    response_text = await page.locator("body").inner_text()
    assert re.search(
        r'"state"\s*:\s*"fail"\s*,\s*"error"\s*:\s*"access forbidden"',
        response_text,
    ), f"期待するJSONデータが表示されていません。actual: {response_text}"

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/external_acc_update/{access_token}」を直接入力する	

404ページが表示されること   
「Resolver404」というテキストが表示されること   

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/external_acc_update/{token}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「https://{admin_domain}/account/login」に対してPOSTメソッドを実行する。

HTTPステータスコードが 302 であること   
レスポンスヘッダーに「Email and/or Password incorrect. Please try again.」というテキストが含まれていること   
※ 正しいメールアドレスとパスワードを送ってもログインせず、エラーメッセージ（「Email and/or Password incorrect. Please try again.」）付きでログイン画面へリダイレクトされます

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    response = await page.request.post(
        f"{admin_rdm_url.rstrip('/')}/account/login",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer": f"{admin_rdm_url.rstrip('/')}/account/login",
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
            "email": idp_username_integrated_admin,
            "password": idp_password_integrated_admin,
        },
        max_redirects=0,
    )

    assert response.status == 302, (f"Expected HTTP status 302, but got {response.status}")

    response_headers = str(response.headers)
    expected_message = "Email and/or Password incorrect. Please try again."
    assert expected_message in response_headers, (
        f"Expected message '{expected_message}' "
        f"not found in response headers: {response_headers}"
    )

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}